In [1]:
import tensorflow as tf

# Enable memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth enabled for GPUs.")
    except RuntimeError as e:
        print(f"RuntimeError: {e}")
else:
    print("No GPUs available.")

import pandas as pd
pd.set_option('display.max_colwidth', None)
import warnings
warnings.filterwarnings("ignore")
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Bidirectional, Embedding, LSTM, SimpleRNN
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold

Memory growth enabled for GPUs.


In [2]:
df = pd.read_csv('../Input/preprocessed.csv', usecols=['class', 'processed_text'])
df.dropna(subset=['processed_text'], inplace=True)
df.dropna(subset=['class'], inplace=True)

In [3]:
X = df['processed_text'].values
y = df['class'].values

In [4]:
tokenizer = Tokenizer(num_words=25000)
tokenizer.fit_on_texts(X)
X_seq = tokenizer.texts_to_sequences(X)
max_length = 500
X_padded = pad_sequences(X_seq, maxlen=max_length, padding='post')
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [5]:
class ProgressLogger(Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"Epoch {epoch+1}: Loss = {logs['loss']:.4f}, Accuracy = {logs['accuracy']:.4f}, "
              f"Val_Loss = {logs['val_loss']:.4f}, Val_Accuracy = {logs['val_accuracy']:.4f}")

In [7]:
input_length = X_padded.shape[1]
num_tokens = 25000
dropout_rate = 0.5
model = Sequential()
model.add(Embedding(input_dim=num_tokens, output_dim=128, input_length=input_length))
model.add(Dropout(dropout_rate))
model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Dropout(dropout_rate))
model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Dropout(dropout_rate))
model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Dropout(dropout_rate))
model.add(SimpleRNN(64, return_sequences=True))
model.add(Dropout(dropout_rate))
model.add(SimpleRNN(64, return_sequences=True))
model.add(Dropout(dropout_rate))
model.add(SimpleRNN(64, return_sequences=False))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []

for fold_num, (train_index, val_index) in enumerate(skf.split(X_padded, y), start=1):
    print(f"Fold {fold_num}:")
    X_train, X_test = X_padded[train_index], X_padded[val_index]
    y_train, y_test = y[train_index], y[val_index]
    
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.001)
    early_stop = EarlyStopping(monitor='val_accuracy', patience=10)
    mc = ModelCheckpoint('../Models/best_dl_model_csv.h5', monitor='val_accuracy', mode='max', verbose=1, save_best_only=True)
    progress_logger = ProgressLogger()
    
    callbacks = [reduce_lr, early_stop, mc, progress_logger]
    
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=128, callbacks=callbacks, verbose=1)
    
    y_pred = (model.predict(X_test) > 0.5).astype(int)
    accuracy_score = accuracy_score(y_test, y_pred)
    fold_accuracies.append(accuracy_score)
    
    print(f"Fold {fold_num} Accuracy: {fold_accuracies[-1]}\n")

Fold 1:
Epoch 1/5
 132/1451 [=>............................] - ETA: 36:01 - loss: 0.7087 - accuracy: 0.4920

KeyboardInterrupt: 